![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 06: Causal Inference in Health Research
***
**Learning objectives**
- Conduct E-value analysis for unmeasured confounding
- Implement Rosenbaum bounds for observational studies
- Perform propensity score trimming and overlap assessment
- Build a comprehensive causal inference reporting template
- Apply placebo tests and refutation checks
- Communicate uncertainty from multiple sensitivity analyses

**Estimated time:** 2.5 hours | **Level:** Advanced | **Libraries:** `numpy`, `scipy`, `sklearn`


## 1. Setup & Dataset

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod06", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42); N = 5000
def sigmoid(x): return 1/(1+np.exp(-x))
age = np.random.normal(60,12,N).clip(30,90)
smoking = np.random.binomial(1,sigmoid(-1+0.02*(age-60)),N)
chd_history = np.random.binomial(1,sigmoid(-2+0.03*(age-60)),N)
ldl_baseline = np.random.normal(140+20*smoking,30,N).clip(60,280)
ses_score = np.random.normal(0,1,N)
statin_logit = -2.0+0.025*(ldl_baseline-140)+1.0*chd_history+0.02*(age-60)-0.3*ses_score
statin = np.random.binomial(1,sigmoid(statin_logit),N)
TRUE_EFFECT = -0.8
mi_logit = -3.0+TRUE_EFFECT*statin+0.03*(age-60)+0.5*smoking+0.8*chd_history+0.005*(ldl_baseline-140)
mi = np.random.binomial(1,sigmoid(mi_logit),N)
df = pd.DataFrame({"age":age,"smoking":smoking,"chd_history":chd_history,
                   "ldl_baseline":ldl_baseline,"ses_score":ses_score,"statin":statin,"mi":mi})
COVARIATES = ["age","smoking","chd_history","ldl_baseline","ses_score"]
print(f"N={N} | Statin={statin.mean()*100:.1f}% | MI={mi.mean()*100:.1f}%")

## 2. E-Value — Unmeasured Confounding Sensitivity

The **E-value** (VanderWeele & Ding, 2017) answers:

> "How strong would an unmeasured confounder have to be (in terms of its association with both treatment and outcome) to fully explain away the observed effect?"

A large E-value means the result is robust to unmeasured confounding.

**Formula for OR:**
```
E-value = OR + sqrt(OR * (OR - 1))     if OR >= 1
E-value = 1/OR + sqrt(1/OR * (1/OR - 1))  if OR < 1
```


In [ ]:
from sklearn.linear_model import LogisticRegression

# Fit adjusted model
lr_adj = LogisticRegression(C=1, max_iter=1000, random_state=42)
lr_adj.fit(df[COVARIATES + ["statin"]], df["mi"])
adj_coef = lr_adj.coef_[0][lr_adj.feature_names_in_.tolist().index("statin")]
adj_or   = np.exp(adj_coef)

def evalue(OR):
    if OR >= 1:
        return OR + np.sqrt(OR * (OR - 1))
    else:
        return 1/OR + np.sqrt(1/OR * (1/OR - 1))

print(f"Adjusted OR for statin: {adj_or:.3f}")
ev = evalue(adj_or)
print(f"E-value: {ev:.3f}")
print()
print("Interpretation:")
print(f"  An unmeasured confounder associated with BOTH statin use AND MI")
print(f"  by a risk ratio of at least {ev:.2f}-fold (in each direction)")
print(f"  would be needed to fully explain away the observed effect.")
print()

# E-values across a range of OR estimates (sensitivity plot)
ors_range = np.linspace(0.3, 0.95, 50)
evalues   = [evalue(o) for o in ors_range]

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(ors_range, evalues, "-", color="#D65F5F", lw=2.5)
axes[0].axvline(adj_or, color="#4878CF", ls="--", lw=1.5, label=f"Our estimate (OR={adj_or:.3f})")
axes[0].axhline(ev,     color="#4878CF", ls=":", lw=1.5, label=f"E-value={ev:.2f}")
axes[0].set_xlabel("Odds Ratio (point estimate)"); axes[0].set_ylabel("E-value")
axes[0].set_title("E-value Sensitivity Curve"); axes[0].legend(fontsize=9)
axes[0].fill_between(ors_range, evalues, 1, alpha=0.1, color="#D65F5F")

# Contour plot: what combinations of confounder-treatment and confounder-outcome RRs explain the effect
rr_range = np.linspace(1, 5, 100)
RRa, RRb = np.meshgrid(rr_range, rr_range)
# OR that unmeasured U would need to induce
explained_or = (RRa * RRb) / (RRa + RRb - 1)
im = axes[1].contourf(rr_range, rr_range, explained_or, levels=20, cmap="RdYlBu_r")
plt.colorbar(im, ax=axes[1], label="OR attributable to U")
axes[1].contour(rr_range, rr_range, explained_or, levels=[adj_or], colors=["black"], linewidths=[2])
axes[1].set_xlabel("RR(U, Treatment)"); axes[1].set_ylabel("RR(U, Outcome)")
axes[1].set_title(f"Unmeasured confounder contour\n(black = combos that explain away OR={adj_or:.3f})")
plt.tight_layout(); plt.savefig("/tmp/mod06/evalue_analysis.png",bbox_inches="tight"); plt.show()


## 3. Overlap & Positivity Assessment

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

# Estimate PS
lr_ps = LogisticRegression(C=1, max_iter=1000, random_state=42)
lr_ps.fit(df[COVARIATES], df["statin"])
df["ps"] = lr_ps.predict_proba(df[COVARIATES])[:,1]

fig, axes = plt.subplots(1,3,figsize=(16,4))

# PS distribution overlap
for grp, color, label in [(0,"#4878CF","Control"),(1,"#D65F5F","Treated")]:
    sub = df[df.statin==grp]["ps"]
    axes[0].hist(sub, bins=40, alpha=0.6, color=color, density=True, label=label)
axes[0].set_xlabel("Propensity score"); axes[0].set_ylabel("Density")
axes[0].set_title("PS Distribution: Overlap Check")
axes[0].legend(fontsize=9)

# Common support region
ps_min_treated = df[df.statin==1]["ps"].min()
ps_max_control = df[df.statin==0]["ps"].max()
in_support = (df["ps"] >= ps_min_treated) & (df["ps"] <= ps_max_control)
pct_support = in_support.mean()*100
axes[0].axvspan(ps_min_treated, ps_max_control, alpha=0.1, color="green", label=f"Common support ({pct_support:.0f}%)")
axes[0].legend(fontsize=8)

# IPTW weight distribution
df["w_ate"] = np.where(df.statin==1, 1/df.ps.clip(0.05,0.95), 1/(1-df.ps.clip(0.05,0.95)))
axes[1].hist(df[df.statin==1]["w_ate"], bins=40, alpha=0.7, color="#D65F5F", label="Treated", density=True)
axes[1].hist(df[df.statin==0]["w_ate"], bins=40, alpha=0.7, color="#4878CF", label="Control", density=True)
axes[1].set_xlabel("IPTW weight"); axes[1].set_ylabel("Density")
axes[1].set_title(f"IPTW Weight Distribution (max={df.w_ate.max():.1f})")
axes[1].legend(fontsize=9)

# Effective sample size vs trimming threshold
trim_thresholds = np.linspace(0.05, 0.40, 30)
ess_vals = []
for t in trim_thresholds:
    df_trim = df[(df.ps >= t) & (df.ps <= 1-t)]
    ess = (len(df_trim)**2) / ((df_trim.statin * 1/df_trim.ps.clip(t,1-t)).sum()**2 / len(df_trim))
    ess_vals.append(ess)
axes[2].plot(trim_thresholds, ess_vals, "-", color="#6ACC65", lw=2)
axes[2].set_xlabel("PS trimming threshold"); axes[2].set_ylabel("Effective sample size")
axes[2].set_title("Effective Sample Size vs Trimming")
plt.tight_layout(); plt.savefig("/tmp/mod06/overlap_positivity.png",bbox_inches="tight"); plt.show()

print(f"Common support: {pct_support:.1f}% of sample in overlapping PS region")
print(f"Extreme weights (>20): {(df.w_ate>20).sum()} patients ({(df.w_ate>20).mean()*100:.1f}%)")


## 4. Placebo Outcome & Refutation Tests

In [ ]:
# Test 1: Placebo outcome — an outcome that should NOT be affected by treatment
# Eye color should not be caused by statin use!
np.random.seed(5)
df["eye_color"] = np.random.binomial(1, 0.35, N)  # purely random

lr_placebo = LogisticRegression(C=1, max_iter=500, random_state=42)
lr_placebo.fit(df[COVARIATES + ["statin"]], df["eye_color"])
placebo_or = np.exp(lr_placebo.coef_[0][lr_placebo.feature_names_in_.tolist().index("statin")])
print(f"Placebo outcome test (eye color ~ statin):")
print(f"  OR = {placebo_or:.3f}  (should be ~1.0)")
print(f"  {'PASS: no spurious effect' if 0.85 < placebo_or < 1.15 else 'FAIL: unexplained association!'}")
print()

# Test 2: Random treatment permutation — break any true causal effect
from sklearn.linear_model import LogisticRegression
np.random.seed(42)
permutation_ors = []
for _ in range(200):
    df_perm = df.copy()
    df_perm["statin_perm"] = np.random.permutation(df["statin"].values)
    lr_perm = LogisticRegression(C=1, max_iter=500, random_state=0)
    lr_perm.fit(df_perm[COVARIATES + ["statin_perm"]], df_perm["mi"])
    idx = lr_perm.feature_names_in_.tolist().index("statin_perm")
    permutation_ors.append(np.exp(lr_perm.coef_[0][idx]))

perm_or_null = np.array(permutation_ors)
p_perm = np.mean(perm_or_null <= adj_or)

print(f"Permutation refutation test:")
print(f"  Null distribution mean OR: {perm_or_null.mean():.3f} (should be ~1.0)")
print(f"  Our observed adjusted OR : {adj_or:.3f}")
print(f"  Permutation p-value      : {p_perm:.4f}")

fig, ax = plt.subplots(figsize=(9,4))
ax.hist(perm_or_null, bins=30, color="#AEC6CF", alpha=0.85, label="Permuted ORs (null)")
ax.axvline(adj_or, color="#D65F5F", ls="--", lw=2.5, label=f"Observed OR ({adj_or:.3f})")
ax.axvline(1.0, color="black", lw=1, label="Null (OR=1)")
ax.set_xlabel("Odds Ratio"); ax.set_ylabel("Count")
ax.set_title(f"Permutation Refutation Test (p={p_perm:.4f})")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod06/permutation_test.png",bbox_inches="tight"); plt.show()


## 5. Clinical Reporting Template

In [ ]:
# Build a comprehensive causal inference report
report = {
    "Study question"   : "Effect of statin use on 30-day MI risk",
    "Design"           : "Observational cohort with propensity score adjustment",
    "N"                : N,
    "Treatment"        : f"Statin use: {statin.mean()*100:.1f}%",
    "Outcome"          : f"MI: {mi.mean()*100:.1f}%",
    "Identification"   : "Backdoor adjustment (Age, Smoking, CHD history, LDL, SES)",
    "Estimator"        : "Adjusted logistic regression + G-computation",
    "Point estimate OR": round(adj_or, 3),
    "E-value"          : round(ev, 2),
    "Common support"   : f"{pct_support:.1f}%",
    "Placebo test"     : f"Eye color OR={placebo_or:.3f} (PASS)" if 0.85<placebo_or<1.15 else "FAIL",
    "Permutation test" : f"p={p_perm:.4f}",
}

print("=" * 60)
print("CAUSAL INFERENCE REPORT")
print("=" * 60)
for key, val in report.items():
    print(f"  {key:25s}: {val}")
print("=" * 60)
print()
print("METHODS NARRATIVE (template):")
print()
print("We estimated the causal effect of statin use on 30-day MI risk")
print("using propensity score-adjusted logistic regression and G-computation.")
print(f"The analysis included N={N} patients; {statin.mean()*100:.1f}% received statins.")
print(f"Propensity scores were estimated using logistic regression on {len(COVARIATES)} pre-treatment covariates.")
print(f"After adjustment, the estimated OR was {adj_or:.3f} (E-value={ev:.2f}),")
print(f"suggesting an unmeasured confounder would need RR>{ev:.1f} with both")
print("treatment and outcome to fully explain the observed association.")
print("Placebo outcome and permutation refutation tests supported the validity of the approach.")


## 6. Knowledge Check
1. E-value = 2.1 for your statin analysis. Is this large or small? What known confounders (e.g. healthy user bias) have RRs in this range?
2. Your PS distribution shows poor overlap for elderly patients (age >80). What are your options?
3. The permutation p-value is 0.001. What does this tell you about the observed association?
4. A placebo outcome (baseline albumin from *before* statin initiation) shows OR=1.12. Should you be concerned?
5. Calculate Rosenbaum's gamma (how much unmeasured confounding would change the conclusion) using a simple sign test.


In [ ]:
# Exercise 5 — simplified Rosenbaum gamma bound
# For a binary outcome, test how much gamma (odds of differential unmeasured confounding)
# would be needed to make the result non-significant

from scipy.stats import binom

# Use matched pairs analysis (simplified)
# In matched pairs, under gamma=1 (no hidden bias), we expect binomial with p=0.5
np.random.seed(42)
# Find pseudo-matched pairs (PS matching)
def simple_match(df, ps_col="ps", treatment="statin"):
    treated = df[df[treatment]==1].sort_values(ps_col).reset_index(drop=True)
    control = df[df[treatment]==0].sort_values(ps_col).reset_index(drop=True)
    n_pairs = min(len(treated), len(control))
    treated_m = treated.iloc[:n_pairs]
    control_m = control.iloc[:n_pairs]
    return treated_m["mi"].values, control_m["mi"].values

y_t, y_c = simple_match(df)
# Discordant pairs (pairs where outcome differs)
disc = (y_t != y_c)
n_disc = disc.sum()
n_treated_wins = ((y_t < y_c) & disc).sum()  # treated has LOWER MI rate
print(f"Discordant pairs: {n_disc} | Treated outcome better: {n_treated_wins}")

# Under gamma=1: test statistic ~ Binomial(n_disc, 0.5)
p_gamma1 = binom.cdf(n_treated_wins, n_disc, 0.5)  # one-sided
print(f"p-value at gamma=1 (no hidden bias): {p_gamma1:.6f}")

# Find gamma that makes p = 0.05
from scipy.optimize import brentq
def p_at_gamma(gamma):
    p_plus = gamma/(1+gamma)
    return binom.cdf(n_treated_wins, n_disc, p_plus) - 0.05

try:
    gamma_threshold = brentq(p_at_gamma, 1, 10)
    print(f"Gamma threshold (Rosenbaum): {gamma_threshold:.2f}")
    print(f"  Unmeasured confounding must change treatment odds by >{gamma_threshold:.2f}-fold")
    print(f"  to make the result non-significant at alpha=0.05")
except:
    print("Result is robust to extreme levels of unmeasured confounding (gamma > 10)")


***
**Next -> NB-08: Capstone — Full Causal Analysis Pipeline**
